# Software Quality Data Warehouse & Web Mining Exploration

This notebook demonstrates exploratory analysis, ETL transformation, Star Schema OLAP queries, Web & Text Mining, Clustering, Association Rules, Time Series Trends, and Graph Mining using the 50K GitHub Archive dataset.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.etl.extract import extract_data
from src.etl.transform import transform_data
from src.warehouse.schema import get_engine, initialize_database
from src.warehouse.queries import olap_rollup, olap_drilldown, olap_slice, olap_dice, olap_pivot
from src.mining.text_mining import extract_top_keywords, get_taxonomy_tree
from src.mining.clustering import cluster_text
from src.mining.association_rules import mine_technology_associations
from src.mining.time_series import compute_hourly_activity, compute_trend_summary
from src.mining.graph_mining import build_developer_repo_graph, compute_network_metrics

## 1. Extract & Inspect Raw Dataset

In [ ]:
raw_df, stats = extract_data(nrows=5000)
print(f"Extracted {stats['row_count']} rows.")
print("Event Type Distribution:", stats["event_distribution"])
raw_df.head(3)

## 2. Transform & Build Star Schema

In [ ]:
enriched_df, tables = transform_data(raw_df)
print("Star Schema Tables:", list(tables.keys()))
for name, tbl in tables.items():
    print(f"{name}: {len(tbl)} records")

## 3. OLAP Operations (Roll-up, Drill-down, Slice, Dice, Pivot)

In [ ]:
engine = get_engine()
print("--- OLAP Roll-up (Hour) ---")
display(olap_rollup(engine, group_by="hour").head(5))

print("--- OLAP Slice (Event Type = IssuesEvent) ---")
display(olap_slice(engine, dimension="event_type", value="IssuesEvent").head(5))

print("--- OLAP Pivot (Category vs Event Type) ---")
display(olap_pivot(engine, row_dimension="category", column_dimension="event_type"))

## 4. Web & Text Mining (TF-IDF & Keywords)

In [ ]:
corpus = enriched_df["extracted_text"].dropna().tolist()
top_keywords = extract_top_keywords(corpus, top_n=10)
pd.DataFrame(top_keywords, columns=["Keyword", "TF-IDF Score"])

## 5. Unsupervised Text Clustering (K-Means)

In [ ]:
cluster_res = cluster_text(corpus[:500], n_clusters=4)
print("Cluster Status:", cluster_res["status"])
print("Cluster Sizes:", cluster_res["cluster_sizes"])
print("Top Terms per Cluster:", cluster_res["top_terms_per_cluster"])

## 6. Association & Episode Rules

In [ ]:
rules = mine_technology_associations(enriched_df, min_support=0.0005, min_confidence=0.05)
print(f"Discovered {len(rules)} technology association rules.")
rules.head(10)

## 7. Time Series Decomposition

In [ ]:
hourly = compute_hourly_activity(enriched_df)
summary = compute_trend_summary(hourly)
print("Trend Summary:", summary)
hourly

## 8. Graph Mining (NetworkX)

In [ ]:
G = build_developer_repo_graph(enriched_df.head(1000))
metrics = compute_network_metrics(G)
print("Network Metrics:", metrics)